# 1.ChatMEssageHistory的使用

它是一个用于存储和管理对话消息的基础类，他直接操作消息对象，是其它记忆组件的底层存储工具

场景1：记忆存储


In [1]:
from langchain_classic.chains.llm import LLMChain
from langchain_classic.memory import ChatMessageHistory
from langchain_core.prompts import PromptTemplate

# 1.ChatMessageHistory的实例化
history = ChatMessageHistory()

#2.添加相关的消息进行存储
history.add_user_message("您好")
history.add_ai_message("很高兴认识你")

#3.打印存储消息
print(history.messages)

D:\conda_envs\python313\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


[HumanMessage(content='您好', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


场景2，对接大模型

In [2]:
import os
import dotenv
from langchain_core.messages import HumanMessage
from langchain_core.messages.ai import AIMessage
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_openai import ChatOpenAI

#1.获取大模型
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0.8,

)

In [3]:
# 1.ChatMessageHistory的实例化
history = ChatMessageHistory()

#2.添加相关的消息进行存储
history.add_user_message("您好")
history.add_ai_message("很高兴认识你")
history.add_user_message("帮我计算1+3+5=？")

response = llm.invoke(history.messages)
print(response.content)

1 + 3 + 5 = 9。


# 2.ConversationBufferMemory

专门用于按原始顺序存储完整的对话历史

举例1以字符串的方式返回

In [4]:
from langchain_classic.memory import ConversationBufferMemory

#1. ConversationBufferMemory实例化
memory = ConversationBufferMemory()

#2.存储相关的消息
#inputs对应的就是用户消息，outputs对应的就是ai消息
memory.save_context(inputs={"input": "你好，我叫小明"}, outputs={"output": "很高兴认识你"})
memory.save_context(inputs={"input": "帮我回答一下：1+2*3=？"}, outputs={"output": "7"})

#3.获取存储的消息
print(memory.load_memory_variables({}))

#说明：返回的字典结构的key叫history

{'history': 'Human: 你好，我叫小明\nAI: 很高兴认识你\nHuman: 帮我回答一下：1+2*3=？\nAI: 7'}


C:\Users\30631\AppData\Local\Temp\ipykernel_1420\270421063.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


举例2以消息列表的方法返回


In [5]:
from langchain_classic.memory import ConversationBufferMemory

#1. ConversationBufferMemory实例化
memory = ConversationBufferMemory(return_messages=True)

#2.存储相关的消息
#inputs对应的就是用户消息，outputs对应的就是ai消息
memory.save_context(inputs={"input": "你好，我叫小明"}, outputs={"output": "很高兴认识你"})
memory.save_context(inputs={"input": "帮我回答一下：1+2*3=？"}, outputs={"output": "7"})

#3.获取存储的消息
#返回消息列表的方式1：
print(memory.load_memory_variables({}))

print("\n")
#返回消息列表的方式2
print(memory.chat_memory.messages)

{'history': [HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='帮我回答一下：1+2*3=？', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


[HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='帮我回答一下：1+2*3=？', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


举例3：结合大模型、提示词模板的使用（PromptTemplate）


In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.llm import LLMChain

llm = ChatOpenAI(
    model_name="gpt-4o-mini",
)

#提示词模板
prompt_template = PromptTemplate.from_template(
    template="""你可以与人类对话
        当前对话: {history}
        人类问题: {question}
        回复:"""
)

#3.提供memory实例
memory = ConversationBufferMemory()

#4.提供Chain
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

response = chain.invoke({"question": "您好，我的名字叫小明"})
print(response)

C:\Users\30631\AppData\Local\Temp\ipykernel_1420\1933599018.py:19: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm,prompt=prompt_template,memory=memory)


{'question': '您好，我的名字叫小明', 'history': '', 'text': '您好，小明！很高兴认识你。有什么我可以帮助你的吗？'}


举例4:基于举例3，显式的设置memory的key值

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.llm import LLMChain

llm = ChatOpenAI(
    model_name="gpt-4o-mini",
)

#提示词模板
prompt_template = PromptTemplate.from_template(
    template="""你可以与人类对话
        当前对话: {chat_history}
        人类问题: {question}
        回复:"""
)

#3.提供memory实例
memory = ConversationBufferMemory(memory_key="chat_history")

#4.提供Chain
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

response = chain.invoke({"question": "您好，我的名字叫小明"})
print(response)

ChatPromptTemplate实例

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_classic.chains.llm import LLMChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import MessagesPlaceholder, HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 2.创建LLM
llm = ChatOpenAI(model_name='gpt-4o-mini')
# 3.创建Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个与人类对话的机器人。"),
    MessagesPlaceholder(variable_name='history'),
    ("human", "问题：{question}")
])
# 4.创建Memory
memory = ConversationBufferMemory(return_messages=True)
# 5.创建LLMChain
llm_chain = LLMChain(prompt=prompt, llm=llm, memory=memory)
# 6.调用LLMChain
res1 = llm_chain.invoke({"question": "中国首都在哪里？"})
print(res1, end="\n\n")

# 3.ConversationChain的使用

In [ ]:
from langchain_classic.chains.conversation.base import ConversationChain
from langchain_core.prompts.prompt import PromptTemplate
from langchain_classic.chains import LLMChain

# 初始化大模型
llm = ChatOpenAI(model="gpt-4o-mini")
template = """以下是人类与AI之间的友好对话描述。AI表现得很健谈，并提供了大量来自其上下文的
具体细节。如果AI不知道问题的答案，它会真诚地表示不知道。
当前对话：{history}
Human: {input}
AI:"""
prompt = PromptTemplate.from_template(template)
# memory = ConversationBufferMemory()
#
# conversation = LLMChain(
# llm=llm,
# prompt = prompt,
# memory=memory,
# verbose=True,
# )
chain = ConversationChain(llm=llm, prompt=prompt, verbose=True)
chain.invoke({"input": "你好，你的名字叫小智"})  #注意，chain中的key必须是input，否则会报错

使用内置默认格式的提示词模版（内部包含input、history变量）

In [ ]:
from langchain_classic.chains.conversation.base import ConversationChain
from langchain_core.prompts.prompt import PromptTemplate
from langchain_classic.chains import LLMChain

# 初始化大模型
llm = ChatOpenAI(model="gpt-4o-mini")
# template = """以下是人类与AI之间的友好对话描述。AI表现得很健谈，并提供了大量来自其上下文的
# 具体细节。如果AI不知道问题的答案，它会真诚地表示不知道。
# 当前对话：
# {history}
# Human: {input}
# AI:"""
# prompt = PromptTemplate.from_template(template)
# memory = ConversationBufferMemory()
#
# conversation = LLMChain(
# llm=llm,
# prompt = prompt,
# memory=memory,
# verbose=True,
# )
chain = ConversationChain(llm=llm)
#内部提供了默认的提示词模板。而此模板中的变量是{input}、{history}
response = chain.invoke({"input": "你好，你的名字叫小智"})  #注意，chain中的key必须是input，否则会报错
print(response)

# 4. ConversationBufferWindowMemory

In [ ]:
from langchain_classic.memory import ConversationBufferWindowMemory
# 1.导入相关包
from langchain_core.prompts.prompt import PromptTemplate
from langchain_classic.chains.llm import LLMChain

# 2.定义模版
template = """以下是人类与AI之间的友好对话描述。AI表现得很健谈，并提供了大量来自其上下文的
具体细节。如果AI不知道问题的答案，它会表示不知道。
当前对话：
{history}
Human: {question}
AI:"""
# 3.定义提示词模版
prompt_template = PromptTemplate.from_template(template)
# 4.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")
# 5.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=1)
# 6.定义LLMChain
conversation_with_summary = LLMChain(
    llm=llm,
    prompt=prompt_template,
    memory=memory,
    verbose=True,
)
# 7.执行链（第一次提问）
respon1 = conversation_with_summary.invoke({"question": "你好，我是孙小空"})
# print(respon1)
# 8.执行链（第二次提问）
respon2 = conversation_with_summary.invoke({"question": "我还有两个师弟，一个是猪小戒，一个是沙小僧"})
# print(respon2)
# 9.执行链（第三次提问）
respon3 = conversation_with_summary.invoke({"question": "我今年高考，竟然考上了1本"})
# print(respon3)
# 10.执行链（第四次提问）
respon4 = conversation_with_summary.invoke({"question": "我叫什么？"})
print(respon4)